In [2]:
import os
import numpy as np
import pandas as pd
import diptest

from sklearn.mixture import GaussianMixture

# ============================================================
# FILE PATHS
# ============================================================

INPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_thickness_summary.csv"

OUTPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_multimodality_summary.csv"

OUTPUT_EXCEL = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_multimodality_summary.xlsx"

# ============================================================
# LOAD DATA
# ============================================================

if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(INPUT_CSV)

df = pd.read_csv(INPUT_CSV)

print("="*80)
print(" GBM THICKNESS MULTIMODALITY ANALYSIS ")
print("="*80)

results = []

# ============================================================
# ANALYSE EACH PATIENT
# ============================================================

for patient_id, group in df.groupby("patient_id"):

    print("\n")
    print("-"*80)
    print(f"Patient : {patient_id}")

    thickness = (
        group["median_thickness_nm"]
        .dropna()
        .values
    )

    n = len(thickness)

    print(f"Number of membranes : {n}")

    if n < 5:
        print("Too few membrane samples.")
        continue

    # --------------------------------------------------------
    # HARTIGAN DIP TEST
    # --------------------------------------------------------

    dip_statistic, dip_p = diptest.diptest(thickness)

    dip_result = (
        "Multimodal"
        if dip_p < 0.05
        else "Unimodal"
    )

    print(f"Dip test p-value : {dip_p:.5f}")

    # --------------------------------------------------------
    # FIT GMM MODELS
    # --------------------------------------------------------

    X = thickness.reshape(-1,1)

    models = {}
    bic_scores = {}
    aic_scores = {}

    for k in [1,2,3]:

        gmm = GaussianMixture(
            n_components=k,
            random_state=42,
            covariance_type="full",
            n_init=20
        )

        gmm.fit(X)

        models[k] = gmm

        bic_scores[k] = gmm.bic(X)

        aic_scores[k] = gmm.aic(X)

    # --------------------------------------------------------
    # BEST MODEL
    # --------------------------------------------------------

    best_components = min(
        bic_scores,
        key=bic_scores.get
    )

    best_model = models[best_components]

    print(f"Best GMM : {best_components} component(s)")

    # --------------------------------------------------------
    # EXTRACT PEAKS
    # --------------------------------------------------------

    peaks = best_model.means_.flatten()

    weights = best_model.weights_.flatten()

    order = np.argsort(peaks)

    peaks = peaks[order]

    weights = weights[order]

    while len(peaks) < 3:
        peaks = np.append(peaks,np.nan)

    while len(weights) < 3:
        weights = np.append(weights,np.nan)

    # --------------------------------------------------------
    # PEAK SEPARATION
    # --------------------------------------------------------

    if np.isnan(peaks[1]):
        separation12 = np.nan
    else:
        separation12 = peaks[1]-peaks[0]

    if np.isnan(peaks[2]):
        separation23 = np.nan
    else:
        separation23 = peaks[2]-peaks[1]

    # --------------------------------------------------------
    # MODEL COMPARISON
    # --------------------------------------------------------

    bic_improvement = bic_scores[1] - bic_scores[best_components]
    aic_improvement = aic_scores[1] - aic_scores[best_components]

    # --------------------------------------------------------
    # CONFIDENCE SCORE
    # --------------------------------------------------------

    confidence_score = 0

    # Dip Test significance
    if dip_p < 0.05:
        confidence_score += 2

    # BIC improvement
    if bic_improvement > 10:
        confidence_score += 2
    elif bic_improvement > 5:
        confidence_score += 1

    # Peak separation
    if not np.isnan(separation12):

        if separation12 > 150:
            confidence_score += 2

        elif separation12 > 75:
            confidence_score += 1

    # Component weights
    valid_weights = weights[~np.isnan(weights)]

    if len(valid_weights) > 1:

        smallest_weight = np.min(valid_weights)

        if smallest_weight >= 0.20:
            confidence_score += 2

        elif smallest_weight >= 0.10:
            confidence_score += 1

    # --------------------------------------------------------
    # CONFIDENCE LABEL
    # --------------------------------------------------------

    if confidence_score >= 7:

        confidence = "Very Strong"

    elif confidence_score >= 5:

        confidence = "Strong"

    elif confidence_score >= 3:

        confidence = "Moderate"

    elif confidence_score >= 1:

        confidence = "Weak"

    else:

        confidence = "Very Weak"

    # --------------------------------------------------------
    # PRINT SUMMARY
    # --------------------------------------------------------

    print(f"Dip result           : {dip_result}")

    print(f"Best GMM             : {best_components}")

    print(f"BIC Improvement      : {bic_improvement:.2f}")

    print(f"AIC Improvement      : {aic_improvement:.2f}")

    print(f"Peak 1               : {peaks[0]:.2f} nm")

    if not np.isnan(peaks[1]):
        print(f"Peak 2               : {peaks[1]:.2f} nm")

    if not np.isnan(peaks[2]):
        print(f"Peak 3               : {peaks[2]:.2f} nm")

    print(f"Peak Separation      : {separation12:.2f} nm"
          if not np.isnan(separation12)
          else "Peak Separation      : N/A")

    print(f"Confidence           : {confidence}")

    # --------------------------------------------------------
    # SAVE RESULTS
    # --------------------------------------------------------

    results.append({

        "Patient_ID": patient_id,

        "Number_of_membranes": n,

        "Dip_statistic":
            round(dip_statistic, 5),

        "Dip_p_value":
            round(dip_p, 5),

        "Dip_result":
            dip_result,

        "Best_GMM_components":
            best_components,

        "AIC_1":
            round(aic_scores[1], 2),

        "AIC_2":
            round(aic_scores[2], 2),

        "AIC_3":
            round(aic_scores[3], 2),

        "BIC_1":
            round(bic_scores[1], 2),

        "BIC_2":
            round(bic_scores[2], 2),

        "BIC_3":
            round(bic_scores[3], 2),

        "Peak_1_nm":
            round(peaks[0], 2),

        "Peak_2_nm":
            None if np.isnan(peaks[1])
            else round(peaks[1], 2),

        "Peak_3_nm":
            None if np.isnan(peaks[2])
            else round(peaks[2], 2),

        "Weight_1":
            round(weights[0], 3),

        "Weight_2":
            None if np.isnan(weights[1])
            else round(weights[1], 3),

        "Weight_3":
            None if np.isnan(weights[2])
            else round(weights[2], 3),

        "Peak_Separation_nm":
            None if np.isnan(separation12)
            else round(separation12, 2),

        "Confidence":
            confidence,

        "Confidence_Score":
            confidence_score
    })
# ============================================================
# PART 3
# SAVE RESULTS & PATIENT RANKING
# ============================================================

summary_df = pd.DataFrame(results)

# ------------------------------------------------------------
# Rank patients
# ------------------------------------------------------------
# Higher evidence of multimodality:
#   1. Dip test significant
#   2. Larger number of GMM components
#   3. Lower BIC

summary_df = summary_df.sort_values(
    by=[
        "Dip_p_value",
        "Best_GMM_components",
        "BIC"
    ],
    ascending=[True, False, True]
).reset_index(drop=True)

summary_df.insert(
    0,
    "Rank",
    np.arange(1, len(summary_df) + 1)
)

# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------

summary_df.to_csv(
    OUTPUT_CSV,
    index=False
)

# ------------------------------------------------------------
# Print ranking
# ------------------------------------------------------------

print("\n")
print("=" * 90)
print("PATIENT MULTIMODALITY RANKING")
print("=" * 90)

print(
    summary_df[
        [
            "Rank",
            "Patient_ID",
            "Dip_p_value",
            "Best_GMM_components",
            "Peak_1_nm",
            "Peak_2_nm",
            "Peak_3_nm"
        ]
    ].to_string(index=False)
)

print("\n")
print("=" * 90)
print("Analysis completed successfully")
print("=" * 90)

print(f"\nResults saved to:\n{OUTPUT_CSV}")

 GBM THICKNESS MULTIMODALITY ANALYSIS 


--------------------------------------------------------------------------------
Patient : 01-24
Number of membranes : 19
Dip test p-value : 0.86254
Best GMM : 3 component(s)
Dip result           : Unimodal
Best GMM             : 3
BIC Improvement      : 32.56
AIC Improvement      : 38.23
Peak 1               : 170.91 nm
Peak 2               : 372.42 nm
Peak 3               : 480.62 nm
Peak Separation      : 201.52 nm
Confidence           : Moderate


--------------------------------------------------------------------------------
Patient : 02-24
Number of membranes : 41
Dip test p-value : 0.99123
Best GMM : 3 component(s)
Dip result           : Unimodal
Best GMM             : 3
BIC Improvement      : 82.24
AIC Improvement      : 92.52
Peak 1               : 241.72 nm
Peak 2               : 785.13 nm
Peak 3               : 1339.96 nm
Peak Separation      : 543.41 nm
Confidence           : Moderate


----------------------------------------------

KeyError: 'BIC'